# Group 9 Project: UC New Bearcat Helper


*Install packages*

In [ ]:
!uv pip install --upgrade openai openai-agents pypdf python-dotenv requests gradio

*Imports and Set-up*

In [ ]:
from dotenv import load_dotenv
import os

from openai import AsyncAzureOpenAI
from agents import (
    Agent,
    Runner,
    function_tool,
    OpenAIChatCompletionsModel,
    set_tracing_disabled,
    SQLiteSession,
    handoff,
)
from agents.extensions.handoff_prompt import RECOMMENDED_PROMPT_PREFIX
import gradio as gr

# Load credentials
load_dotenv(override=True)

AZURE_OPENAI_ENDPOINT = os.getenv("AZURE_OPENAI_ENDPOINT")
AZURE_OPENAI_API_KEY = os.getenv("AZURE_OPENAI_API_KEY")
DEPLOYMENT = "gpt-5.4-ptu"

print("Endpoint loaded:", bool(AZURE_OPENAI_ENDPOINT))
print("API key loaded:", bool(AZURE_OPENAI_API_KEY))

# Configure OpenAI SDK
set_tracing_disabled(True)

azure_client = AsyncAzureOpenAI(
    azure_endpoint=AZURE_OPENAI_ENDPOINT,
    api_key=AZURE_OPENAI_API_KEY,
    api_version="2024-10-21",
)

azure_model = OpenAIChatCompletionsModel(
    model=DEPLOYMENT,
    openai_client=azure_client,
)

print("Azure model configured for the OpenAI Agents SDK.")

1. UC Directory Agent and Tools

In [ ]:
reader = PdfReader("notes/directory.pdf")
directory_notes = ""
for page in reader.pages:
    text = page.extract_text()
    if text:
        notes += text

In [ ]:
@function_tool
def uc_directory_search(topic: str) -> str:
    """Look up a department in the UC directory and
    return the department's contact information and website.
    
    Args:
        topic (str): The topic to search for in the UC directory.
    """
    print(f"Tool called to look up the department for: {topic}")
    key = topic.strip().lower()
    if key not in directory_notes:
        return f"We could not find anything related to '{topic}' in the directory."
    info = directory_notes[key]

In [ ]:
uc_directory_instructions = f"""{RECOMMENDED_PROMPT_PREFIX}
You are a helpful assistant that can search the UC directory for information.

The user will provide a question that has the key words related to the
topics column listed in the notes/directory.pdf, you will look up a department
in the UC directory and return the appropriate contact information.

If the user ask about a specific department, you will return the contact
information for the department.

If the user ask about a department not listed in the PDF, direct them to contact
the Automated Directory Attendant at (513) 556-6000.

"""
uc_directory_agent = Agent(
    instructions=uc_directory_instructions,
    model=azure_model,
    tools=[function_tool(uc_directory_search)],
)